In [ ]:



# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
ollama_url = "http://localhost:11434/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url) if anthropic_api_key else None
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

requests.get("http://localhost:11434/").content


!ollama pull llama3.2



# Models

gpt_model = "gpt-4.1-mini"
claude_model = "claude-haiku-4-5"
llama_model = "llama3.2"


# Shared conversation
conversation = """
Alex: Hi there
Blake: Hi
"""


# -----------------------------
# ALEX = GPT
# -----------------------------

def call_alex():

    system_prompt = """
You are Alex, a chatbot who is very argumentative; you disagree with anything in the conversation and you challenge everything, in a snarky way.
You are in a conversation with Blake and Charlie.
"""

    user_prompt = f"""
You are Alex, in conversation with Blake and Charlie.
The conversation so far is as follows:
{conversation}

Now with this, respond with what you would like to say next, as Alex.
"""

    response = openai.chat.completions.create(
        model=gpt_model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content


# -----------------------------
# BLAKE = CLAUDE
# -----------------------------

def call_blake():

    system_prompt = """
You are Blake, a very polite and courteous chatbot.
You try to agree with the others or find common ground.
You are in a conversation with Alex and Charlie.
"""

    user_prompt = f"""
You are Blake, in conversation with Alex and Charlie.
The conversation so far is as follows:
{conversation}

Now with this, respond with what you would like to say next, as Blake.
"""

    response = anthropic.chat.completions.create(
        model=claude_model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content


# -----------------------------
# CHARLIE = LLAMA
# -----------------------------

def call_charlie():

    system_prompt = """
You are Charlie, a neutral and balanced chatbot.
You listen to both sides and give your own opinion.
You are in a conversation with Alex and Blake.
"""

    user_prompt = f"""
You are Charlie, in conversation with Alex and Blake.
The conversation so far is as follows:
{conversation}

Now with this, respond with what you would like to say next, as Charlie.
"""

    response = ollama.chat.completions.create(
        model=llama_model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content


# -----------------------------
# RUN CONVERSATION
# -----------------------------

display(Markdown(f"### Conversation\n{conversation}"))

for i in range(5):

    # Alex speaks
    alex_next = call_alex()
    conversation += f"\nAlex: {alex_next}"
    display(Markdown(f"### Alex\n{alex_next}"))

    # Blake sees Alex's latest message
    blake_next = call_blake()
    conversation += f"\nBlake: {blake_next}"
    display(Markdown(f"### Blake\n{blake_next}"))

    # Charlie sees both Alex and Blake's latest messages
    charlie_next = call_charlie()
    conversation += f"\nCharlie: {charlie_next}"
    display(Markdown(f"### Charlie\n{charlie_next}"))